# Day 11 - Word Embeddings and Semantic Similarity


### Importing Libraries and Loading the Model 

In [1]:
import sys

# Wrapping in quotes to handle any spaces in the file path!
!"{sys.executable}" -m pip install sentence-transformers


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: C:\Users\K Shridharan\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import spacy
from sentence_transformers import SentenceTransformer, util

# Load the NLP tools
print("Loading models (this might take a few seconds)...")
nlp = spacy.load("en_core_web_sm")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Models loaded successfully!")

# Load our dataset
df = pd.read_csv("clean_jobs.csv")
df['clean_description'] = df['clean_description'].fillna("")

Loading models (this might take a few seconds)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\K Shridharan\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\K Shridharan\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Models loaded successfully!


### Proving the Concept (The "ML" Problem)

In [3]:
# Define our target skill and a few variations/unrelated terms
target_skill = "Machine Learning"
term_1 = "ML"
term_2 = "Artificial Intelligence"
term_3 = "Data Entry"

# Convert the text into dense mathematical vectors (embeddings)
target_emb = model.encode(target_skill)
emb_1 = model.encode(term_1)
emb_2 = model.encode(term_2)
emb_3 = model.encode(term_3)

# Calculate Cosine Similarity (1.0 means exact match, 0.0 means completely unrelated)
print(f"Comparing against target: '{target_skill}'\n" + "-"*40)
print(f"Similarity with '{term_1}': {util.cos_sim(target_emb, emb_1).item():.4f} (High Match)")
print(f"Similarity with '{term_2}': {util.cos_sim(target_emb, emb_2).item():.4f} (Moderate Match)")
print(f"Similarity with '{term_3}': {util.cos_sim(target_emb, emb_3).item():.4f} (Low Match)")

Comparing against target: 'Machine Learning'
----------------------------------------
Similarity with 'ML': 0.3727 (High Match)
Similarity with 'Artificial Intelligence': 0.7035 (Moderate Match)
Similarity with 'Data Entry': 0.3535 (Low Match)


### Building the Semantic Extractor

In [4]:
def semantic_skill_search(text, target_skills, threshold=0.55):
    """
    Chunks a job description and uses embeddings to find skills that 
    semantically match the target dictionary.
    """
    doc = nlp(str(text).lower())
    
    # Extract noun chunks (short phrases) from the text to compare
    candidates = list(set([chunk.text for chunk in doc.noun_chunks]))
    
    if not candidates:
        return []
        
    # Generate embeddings for both our dictionary and the text chunks
    skill_embs = model.encode(target_skills)
    candidate_embs = model.encode(candidates)
    
    # Calculate the similarity matrix
    cosine_scores = util.cos_sim(skill_embs, candidate_embs)
    
    found_skills = set()
    
    # Check if any chunk in the text strongly matches our target skills
    for i, skill in enumerate(target_skills):
        for j, candidate in enumerate(candidates):
            if cosine_scores[i][j].item() > threshold:
                found_skills.add(skill)
                
    return list(found_skills)

print("Semantic Search Engine built successfully!")

Semantic Search Engine built successfully!


### Final Step is Testing the Engine

### Test 1

In [5]:
# A recruiter's master skill dictionary
tech_dictionary = ["Machine Learning", "Natural Language Processing", "Amazon Web Services"]

# A tricky job description missing exact matches
sample_jd = "We are hiring an engineer with deep expertise in ML, neural networks, and deploying models on AWS."

print(f"Target Skills to Find: {tech_dictionary}")
print(f"Job Description: '{sample_jd}'\n")

# Run the semantic search!
extracted = semantic_skill_search(sample_jd, tech_dictionary)
print(f"Semantically Extracted Skills: {extracted}")

Target Skills to Find: ['Machine Learning', 'Natural Language Processing', 'Amazon Web Services']
Job Description: 'We are hiring an engineer with deep expertise in ML, neural networks, and deploying models on AWS.'

Semantically Extracted Skills: ['Machine Learning', 'Amazon Web Services']


### Test 2: 

In [16]:
# The JD lists specific frameworks, but the dictionary only has broad concepts.
jd_1 = "Looking for a developer capable of building scalable frontends in React and REST APIs using Express and Node.js."
dict_1 = ["Full Stack Web Development", "MERN Stack", "Graphic Design"]
print(f"\nTest 2: Web Stack Context")
print(f"Target Dictionary: {dict_1}")
print(f"Extracted: {semantic_skill_search(jd_1, dict_1, threshold=0.35)}")


Test 2: Web Stack Context
Target Dictionary: ['Full Stack Web Development', 'MERN Stack', 'Graphic Design']
Extracted: ['Graphic Design', 'Full Stack Web Development']


### Test 3: 

In [17]:
# Testing if it understands the relationship between specific AI models and the broader field.
jd_2 = "Must have hands-on experience deploying YOLOv8 models, working with CNNs, and building real-time image analysis pipelines."
dict_2 = ["Computer Vision", "Object Detection", "Blockchain Technology"]
print(f"\nTest 3: Computer Vision Context")
print(f"Target Dictionary: {dict_2}")
print(f"Extracted: {semantic_skill_search(jd_2, dict_2, threshold=0.35)}")


Test 3: Computer Vision Context
Target Dictionary: ['Computer Vision', 'Object Detection', 'Blockchain Technology']
Extracted: ['Object Detection', 'Computer Vision']


### Test 4: 

In [18]:
# Testing if it connects specific services to broader cloud engineering concepts.
jd_3 = "The role requires managing S3 buckets, configuring IAM policies, and maintaining CI/CD deployment pipelines."
dict_3 = ["AWS Cloud Ecosystem", "DevOps Engineering", "Social Media Marketing"]
print(f"\nTest 4: Cloud Infrastructure Context")
print(f"Target Dictionary: {dict_3}")
print(f"Extracted: {semantic_skill_search(jd_3, dict_3, threshold=0.35)}")


Test 4: Cloud Infrastructure Context
Target Dictionary: ['AWS Cloud Ecosystem', 'DevOps Engineering', 'Social Media Marketing']
Extracted: ['DevOps Engineering', 'AWS Cloud Ecosystem']


### Test 5:

In [19]:
# Testing if it survives messy recruiter phrasing and typos.
jd_4 = "Need a data wiz who knows how to wrok with structured query langauge and relational dbases."
dict_4 = ["SQL", "Database Administration", "Cybersecurity"]
print(f"\nTest 5: Messy Phrasing & Typos")
print(f"Target Dictionary: {dict_4}")
print(f"Extracted: {semantic_skill_search(jd_4, dict_4, threshold=0.35)}")
# Should still find SQL and Database Admin despite the messy text.

print("\n" + "="*50 + "\nStress Testing Complete!")


Test 5: Messy Phrasing & Typos
Target Dictionary: ['SQL', 'Database Administration', 'Cybersecurity']
Extracted: ['Database Administration', 'SQL']

Stress Testing Complete!
